# 07. 进程与系统资源

AGDR VLA 入门教程 · 第 2 章 · 第 2 周

- **学习目标**：能查看和结束进程，管理前后台任务，查看 CPU、内存、硬盘和 GPU 的使用情况
- **前置知识**：[04-管道与重定向](./04-管道与重定向.ipynb)
- **预计学习时长**：1 小时

## 进程是什么

程序运行起来就是**进程**，每个进程都有一个唯一编号 **PID**。训练卡死、端口被占用、GPU 显存没释放，通常都要先找到对应的进程。

| 命令 | 作用 |
| --- | --- |
| `ps aux` | 列出所有进程 |
| `ps aux \| grep python` | 找出名字里带 python 的进程 |
| `pgrep -a python` | 按名字查找，直接显示 PID 和命令 |
| `top` | 实时查看，按 `q` 退出 |
| `htop` | 更好用的 `top`，需要安装：`sudo apt install htop` |

`ps aux` 中常看这几列：`USER` 所有者，`PID` 编号，`%CPU`、`%MEM` 占用，`COMMAND` 命令。

In [ ]:
%%bash
ps aux | head -n 5

请在终端里运行一次 `top`。按 `M` 可以按内存排序，按 `P` 可以按 CPU 排序，按 `q` 退出。

## 结束进程

`kill` 的本质是给进程发**信号**：

| 操作 | 信号 | 含义 |
| --- | --- | --- |
| `Ctrl + C` | SIGINT (2) | 中断前台程序 |
| `kill <PID>` | SIGTERM (15) | 请求退出，程序可以保存数据、清理后再退出 |
| `kill -9 <PID>` | SIGKILL (9) | 强制结束，程序来不及做任何清理 |
| `pkill <名字>` | SIGTERM (15) | 按名字结束，可能误伤同名进程 |

**先用 `kill`，等一会儿没反应再用 `kill -9`。** 直接 `kill -9` 可能损坏正在写入的文件。

In [ ]:
%%bash
sleep 60 > /dev/null 2>&1 &
pid=$!
echo "启动了后台进程，PID=$pid"
ps -p "$pid" -o pid,comm
kill "$pid"
sleep 0.2
ps -p "$pid" > /dev/null || echo "进程 $pid 已结束"

`&` 让命令在后台运行，`$!` 保存最近一个后台进程的 PID。

## 前台与后台

| 操作 | 作用 |
| --- | --- |
| `命令 &` | 直接在后台运行 |
| `Ctrl + Z` | **暂停**前台任务（注意：没有结束，也没有继续运行） |
| `bg` | 让暂停的任务在后台继续 |
| `fg` | 把任务调回前台 |
| `jobs` | 查看当前终端的任务 |

请在终端里试一遍：

```bash
sleep 300      # 按 Ctrl + Z 暂停
jobs           # 显示 Stopped
bg             # 在后台继续
jobs           # 显示 Running
fg             # 回到前台，再按 Ctrl + C 结束
```

### 关掉终端，任务怎么办

普通的后台任务，可能会随着终端关闭而被结束。要让任务在终端关闭后继续运行：

```bash
nohup python train.py > train.log 2>&1 &
```

`nohup` 让程序忽略终端关闭时的挂断信号（SIGHUP）。第 09 节的 `tmux` 通常更好用，也是长时间训练的推荐做法。

## 查看系统资源

| 资源 | 命令 | 看什么 |
| --- | --- | --- |
| 硬盘容量 | `df -h` | 各分区还剩多少空间 |
| 目录大小 | `du -sh <目录>` | 某个目录占了多少 |
| 内存 | `free -h` | 主要看 `available` 列 |
| CPU 核数 | `nproc` | 决定 DataLoader 的 worker 数时有用 |
| GPU | `nvidia-smi` | 显存、利用率、占用 GPU 的进程 |

In [ ]:
%%bash
df -h /
echo ---
free -h
echo ---
nproc
echo ---
du -sh /etc 2>/dev/null

硬盘满了，一层层找出占空间的目录：

```bash
du -h --max-depth=1 ~ | sort -rh | head -n 10
```

`sort -h` 能正确比较 `1.5G` 和 `300M` 这样的大小。

## nvidia-smi：查看 GPU

```bash
nvidia-smi              # 查看一次
watch -n 1 nvidia-smi   # 每秒刷新，按 Ctrl + C 退出
```

需要看的信息：

- **Memory-Usage**：显存用量。CUDA out of memory 时先看这里
- **GPU-Util**：GPU 利用率。长期很低时，瓶颈可能在数据加载
- **Processes**：哪些进程在占用 GPU。可以把这里的 PID 用于 `ps` 或 `kill`

> **虚拟机一般用不了 GPU**，运行 `nvidia-smi` 报 command not found 是正常的。先记住命令，到服务器或双系统上再用。
>
> 在共享服务器上，只能 `kill` 自己的进程。**先确认 `USER` 那一列是你自己。**

## 谁占了这个端口

启动 Jupyter、ROS2 工具或其他网络服务时，可能遇到 `Address already in use`：

```bash
ss -ltnp | grep 8888
```

`-l` 监听中，`-t` TCP，`-n` 显示数字端口，`-p` 显示进程（别人的进程要 `sudo` 才能看到）。找到 PID 后，确认是自己的程序，再决定是否结束。

## 动手试试

在终端里完成，不计分：

1. 在后台运行 `sleep 600`，用 `pgrep -a sleep` 找到它的 PID，再用 `kill` 结束
2. 查看 `/` 所在分区还剩多少空间
3. 找出主目录下最大的 3 个子目录

<details>
<summary>参考答案</summary>

```bash
sleep 600 &
pgrep -a sleep
kill <上一步看到的 PID>
df -h /
du -h --max-depth=1 ~ | sort -rh | head -n 4
```

第 3 题取 4 行，因为第一行是主目录自身的合计。

</details>

## 小结

| 你应该能 | 相关内容 |
| --- | --- |
| 用 `ps`、`pgrep`、`top` 找进程 | 进程是什么 |
| 区分 `kill` 和 `kill -9`，按正确顺序使用 | 结束进程 |
| 管理前后台任务，说出 `nohup` 的作用 | 前台与后台 |
| 查看硬盘、内存、CPU 和 GPU | 查看系统资源 |

下一节：[08-软件安装与压缩包](./08-软件安装与压缩包.ipynb)